In [ ]:
using LinearAlgebra
using BSplineKit
using PyCall
using DelimitedFiles
using Plots
using NonlinearEigenproblems
include("BaseFlow_cavity.jl")
include("Stability_Cavity.jl")

In [ ]:
Res = 1000
N_cheb = 129
mode = 1
Ro = - 1.0
Co = 2 - Ro - Ro^2
Ts = 0
u0,v0,w0,du0,dv0,x = CRC_BF.BaseFlow(Res,Ro,Ts,mode)
D,D2,z = CRC_BF.Cheb(N_cheb,mode)
F,G,H = CRC_BF.interp(u0,v0,w0,z,N_cheb,mode)

In [ ]:
R = 300
be = 0.1
OMEGA = 0.0
omega = OMEGA/R
c = 0.4
cof = CRC_STA.Spatial_mode_BEK1((F),(G.-1),(H),R,N_cheb,D,D2,Res)
L0_raw,L1_raw,L2_raw= CRC_STA.assemble_mat(cof,D,D2,be,omega,R)
L0,L1,L2 = CRC_STA.boudary_condition(L0_raw,L1_raw,L2_raw,N_cheb,mode)
nep = PEP([L0,L1,L2]);
eigval,eigvec = iar(nep, σ = c, neigs = 3,maxit = 500 , tol=1e-14)
vel = CRC_STA.eig_full(eigvec,N_cheb,1)
@show eigval

In [ ]:
function interation(R, alpha, be_ini)
    data_all = [0 0 0]
    cof = CRC_STA.Spatial_mode_BEK1((F),(G.-1),(H),R,N_cheb,D,D2,Res)
    H0,H1 = CRC_STA.assemble_time_mat(cof,D,D2,be_ini,alpha,R,N_cheb)
    C = eigen(H0,H1)
    val_all = C.values
    vec_all = C.vectors
    val_all = filter(!isnan,val_all)
    omega_differ = 0.0/R
    target_mode = findmin(abs.(val_all.- omega_differ))
    val = val_all[target_mode[2]]
    vec = vec_all[:,target_mode[2]]
    for be = 0.0 : 0.001 : 0.3
        sigma = val
        q0 = vec
        H0,H1 = CRC_STA.assemble_time_mat(cof,D,D2,be,alpha,R,N_cheb)
        val,vec = rayleigh_quotient_iteration(H0,H1,sigma; q0)
        data_all = [data_all; [alpha be val]]
        writedlm("test.dat",data_all)
    end
end

In [ ]:
interation(500, 0.4, 0.0)

In [ ]:
R = 500
be = 0.01
alpha = 0.4
cof = CRC_STA.Spatial_mode_BEK1((F),(G.-1),(H),R,N_cheb,D,D2,Res)
H0,H1 = CRC_STA.assemble_time_mat(cof,D,D2,be,alpha,R,N_cheb)
C = eigen(H0,H1)
val_all = C.values
vec_all = C.vectors
val_all = filter(!isnan,val_all)
omega_differ = 0.0/R
# target_mode = findmin(abs.(val_all.- omega_differ))
target_mode = findmin(filter(x -> x > 0, imag.(val_all)))
# val = val_all[target_mode[2]]
# vec = vec_all[:,target_mode[2]]
# data_all = [data_all; [alpha be val]]
    # writedlm("test.dat",data_all)

In [ ]:
val_all[16]

In [ ]:
imag.(val_all)

In [ ]:
scatter(real.(val_all),imag.(val_all),xlims=[-0.1,0.1],ylims=[-0.1,0.1])

In [ ]:
function rayleigh_quotient_iteration(A, B, sigma; q0=rand(size(A, 1), 1))

    flg = true
    i=1
    while flg
        i=i+1
        sigma0 = real(sigma[1]) + abs(imag(sigma[1]))im + 0.0e0im
        q = (A - sigma*B) \ (B*q0)
        q0 = q/maximum(abs.(q))
        sigma = ((q0'*(A*q0))/(q0'*(B*q0)))[1]
        if abs(sigma-sigma0)<=eps(1.0f0)
            flg = false
        end
        if i==20
            flg=false
        end
    end

      return sigma, q0
end